# 4주차 예제 — 따릉이 대여량 예측 (Random Forest + 모델 비교)

2~3주차와 같은 데이터·변수를 그대로 이어갑니다. 이번 주는 Random Forest를 추가하고, **선형회귀 / 단일 트리 / Random Forest 세 모델을 한 표에서 비교**합니다.

1. Random Forest란 무엇인가 (간단한 예제로 원리부터)
2. 지난주 데이터·모델 재구성
3. Random Forest 학습
4. Feature Importance 비교 (단일 트리 vs RF)
5. 세 모델 종합 비교


## Part 1. Random Forest란 무엇인가 (간단한 예제)

3주차의 기온→아이스크림 판매량 소규모 예제 데이터를 다시 사용합니다. 먼저 단일 트리가 데이터 변화에 얼마나 민감한지 확인합니다. **관측치가 한두 개만 달라져도 트리 구조와 예측값이 크게 달라질 수 있습니다.**


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False


In [ ]:
toy = pd.DataFrame({
    '기온': [5, 10, 15, 20, 25, 30],
    '판매량': [20, 35, 40, 55, 70, 90],
})

tree_all = DecisionTreeRegressor(max_depth=2, random_state=42).fit(toy[['기온']], toy['판매량'])
toy_drop = toy.drop(index=3)  # 기온=20, 판매량=55인 점 하나만 빼봅니다
tree_drop = DecisionTreeRegressor(max_depth=2, random_state=42).fit(toy_drop[['기온']], toy_drop['판매량'])

print('전체 6개로 학습:')
print(export_text(tree_all, feature_names=['기온']))
print('점 1개를 뺀 5개로 학습:')
print(export_text(tree_drop, feature_names=['기온']))


In [ ]:
print('기온=22일 때 예측값 비교')
print('전체 데이터로 학습한 트리:', tree_all.predict(pd.DataFrame({'기온': [22]}))[0])
print('점 1개 뺀 데이터로 학습한 트리:', tree_drop.predict(pd.DataFrame({'기온': [22]}))[0])


6개 관측치 중 1개를 제외하자 예측값이 55.0에서 70.0으로 바뀝니다. **트리는 데이터의 작은 변화에도 구조 전체가 달라질 수 있습니다.** 이러한 성질을 '분산(variance)이 크다'고 표현합니다. 이제 같은 두 데이터셋으로 Random Forest를 학습해 변화 폭을 비교합니다.


In [ ]:
rf_all = RandomForestRegressor(n_estimators=100, max_depth=2, random_state=42).fit(toy[['기온']], toy['판매량'])
rf_drop = RandomForestRegressor(n_estimators=100, max_depth=2, random_state=42).fit(toy_drop[['기온']], toy_drop['판매량'])

print('전체 데이터 RF 예측(기온=22):', rf_all.predict(pd.DataFrame({'기온': [22]}))[0])
print('점 1개 뺀 RF 예측(기온=22) :', rf_drop.predict(pd.DataFrame({'기온': [22]}))[0])


단일 트리는 예측값이 15만큼(55→70) 바뀌었고, Random Forest는 약 3만큼(59.5→62.8) 바뀝니다. **Random Forest는 각 트리마다 행을 다르게 부트스트랩 표집하고, `max_features` 설정에 따라 분기 후보 변수도 제한할 수 있습니다.** 이 예제는 입력 변수가 기온 하나뿐이므로 트리의 차이는 주로 행 표집에서 생깁니다. 개별 트리에는 여전히 변동이 있지만, 여러 예측을 평균하면 각 트리의 '흔들림'이 일부 상쇄되어 더 안정적인 예측을 얻을 수 있습니다. 이것이 **앙상블(ensemble)** 의 핵심 아이디어입니다.


## Part 2. 지난주 데이터·모델 재구성

2~3주차와 동일한 전처리와 모델입니다.

원본 데이터는 [서울시 공공자전거 따릉이 이용현황(일별 대여건수)](https://data.seoul.go.kr/dataList/OA-14994/A/1/datasetView.do) 페이지에서 기간별 파일을 확인하고 내려받을 수 있습니다. 내려받은 파일을 `../dataset/open/extracted/따릉이 공공데이터/02_이용정보/` 폴더에 두고 `서울특별시 공공자전거 일별 대여건수_*.csv` 파일명을 유지하면 아래 코드를 그대로 실행할 수 있습니다.


In [ ]:
import glob
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

base = '../dataset/open/extracted/따릉이 공공데이터/02_이용정보'
files = sorted(glob.glob(base + '/서울특별시 공공자전거 일별 대여건수_*.csv'))
dfs = [pd.read_csv(f, encoding='cp949') for f in files]
df = pd.concat(dfs, ignore_index=True)
df['대여일자'] = pd.to_datetime(df['대여일자'])
df = df.sort_values('대여일자').reset_index(drop=True)

df['day_index'] = (df['대여일자'] - df['대여일자'].min()).dt.days
df['월'] = df['대여일자'].dt.month
df['요일'] = df['대여일자'].dt.day_name()

month_dummies = pd.get_dummies(df['월'], prefix='월', drop_first=True)
weekday_dummies = pd.get_dummies(df['요일'], prefix='요일', drop_first=True)
X = pd.concat([df[['day_index']], month_dummies, weekday_dummies], axis=1)
y = df['대여건수']

split_idx = len(df) - 60
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]


In [ ]:
linear_model = LinearRegression().fit(X_train, y_train)
rmse_linear = root_mean_squared_error(y_test, linear_model.predict(X_test))

tree_model = DecisionTreeRegressor(max_depth=6, min_samples_leaf=20, random_state=42).fit(X_train, y_train)
rmse_tree = root_mean_squared_error(y_test, tree_model.predict(X_test))

print('선형회귀 RMSE:', rmse_linear)
print('단일 트리(가지치기) RMSE:', rmse_tree)


## Part 3. Random Forest 학습

Part 1에서 확인한 원리를 그대로, 실제 데이터에 적용합니다.


In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=100,   # 트리 개수
    max_depth=6,
    min_samples_leaf=10,
    random_state=42,
)
rf_model.fit(X_train, y_train)

rmse_rf_train = root_mean_squared_error(y_train, rf_model.predict(X_train))
rmse_rf_test = root_mean_squared_error(y_test, rf_model.predict(X_test))
print('Random Forest - train RMSE:', rmse_rf_train)
print('Random Forest - test RMSE :', rmse_rf_test)


Random Forest의 test RMSE를 단일 트리(가지치기)의 결과와 비교합니다. 여러 트리의 예측을 평균한 뒤 RMSE가 낮아졌다면, Part 1에서 살펴본 분산 감소 원리가 실제 데이터에서도 나타난 것으로 해석할 수 있습니다.


## Part 4. Feature Importance 비교 — 단일 트리 vs Random Forest

RF의 feature importance는 트리 100개의 importance를 평균 낸 값입니다. 단일 트리보다 특정 변수에 집중되는 정도가 줄어들 수 있으며, 데이터의 작은 변화에 덜 민감한 중요도를 보여주는 경향이 있습니다.


In [ ]:
importance_compare = pd.DataFrame({
    '단일 트리': tree_model.feature_importances_,
    'Random Forest': rf_model.feature_importances_,
}, index=X.columns).sort_values('Random Forest', ascending=False)

fig, ax = plt.subplots(figsize=(8, 6))
importance_compare.head(10).plot(kind='barh', ax=ax)
ax.invert_yaxis()
ax.set_title('Feature Importance 비교 (상위 10개)')
plt.show()


## Part 5. 세 모델 종합 비교

선형회귀 / 단일 트리(가지치기) / Random Forest, 세 모델의 test RMSE를 한 표와 그래프로 비교합니다.


In [ ]:
comparison = pd.DataFrame({
    '모델': ['선형회귀', '단일 트리(가지치기)', 'Random Forest'],
    'test RMSE': [rmse_linear, rmse_tree, rmse_rf_test],
}).sort_values('test RMSE')
comparison


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(df['대여일자'].iloc[split_idx:], y_test.values, label='실제', linewidth=2)
ax.plot(df['대여일자'].iloc[split_idx:], linear_model.predict(X_test), label='선형회귀')
ax.plot(df['대여일자'].iloc[split_idx:], tree_model.predict(X_test), label='단일 트리')
ax.plot(df['대여일자'].iloc[split_idx:], rf_model.predict(X_test), label='Random Forest')
ax.legend()
ax.set_title('실제 vs 세 모델 예측')
plt.xticks(rotation=45)
plt.show()


**해석**

- Random Forest는 단일 트리보다 test RMSE가 낮습니다. Part 1에서 확인한 앙상블 효과로 단일 트리의 분산(불안정성)을 줄인 결과입니다.
- 다만 Random Forest의 test RMSE는 선형회귀보다 높게 나타납니다. 여전히 트리를 기반으로 하기 때문에 3주차에서 확인한 것과 같은 한계, 즉 **훈련에서 본 `day_index` 범위 밖으로 추세를 연장(외삽)하지 못하는 문제**가 남아 있기 때문입니다.
- 따라서 앙상블은 모든 데이터에서 한 모델을 항상 더 우수하게 만드는 방법이라기보다, **트리 계열 모델의 분산을 줄이는 데 도움을 주며 외삽과 같은 구조적 한계는 별도로 살펴볼 필요가 있습니다.**


## 인사이트 정리 (예시)

- Random Forest의 test RMSE는 단일 트리보다 낮습니다. 이는 앙상블이 분산을 줄여 일반화 성능을 높인 결과이며, 간단한 예제에서 확인한 '흔들림이 작아지는' 현상과 같은 원리입니다.
- 선형회귀는 세 모델 중 test RMSE가 가장 낮습니다. 이 문제처럼 뚜렷한 추세가 있는 시계열에서는 외삽이 가능한 모델이 유리할 수 있습니다.
- Feature importance는 단일 트리보다 RF에서 더 안정적으로 나타납니다. 여러 트리의 결과를 평균하므로 데이터의 작은 변화에 덜 민감한 경향이 있습니다.

과제(물류 유통량 예측)에서는 시계열 추세가 없는 데이터로 같은 세 모델을 비교합니다. 이번 예제와 다른 순위가 나타날 수 있으며, **모델 비교의 결론은 데이터의 구조와 평가 방법에 따라 달라진다**는 점을 결과 표에서 확인할 수 있습니다.
